# Exercise 1: Retention-basin information extraction

Unstructured reports and papers often mention flood-retention basins, polders, and reservoirs without a structured database provided. This exercise uses **DLLMForge** package and **Azure OpenAI model** to extract retention basin information from PDF, and output a structured JSON that matches a user-defined schema, which can further be used for your own modeling or analysis works.

You will:

1. Define and inspect a hand-written Pydantic schema.
2. Convert a PDF with Docling (sections, markdown tables, `[PAGE N]` tags).
3. Extract records with `DoclingInfoExtractor`.
4. Compare the JSON to 'ground truth' values in `README.md` for Eupen and La Gileppe.

Prerequisites: course `uv sync` / hydrology-env verify is green, and Azure OpenAI variables are in the repo-root `.env`. Never paste API keys into this notebook.

## Step 1: Imports and initialize LLMs


In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from dllmforge.IE_agent_extractor_docling import DoclingDocumentProcessor, DoclingInfoExtractor
from dllmforge.langchain_api import LangchainAPI

EXERCISE_DIR = Path.cwd()
if not (EXERCISE_DIR / "schema.py").exists():
    EXERCISE_DIR = Path.cwd() / "exercises" / "01_information_extraction"
if str(EXERCISE_DIR) not in sys.path:
    sys.path.insert(0, str(EXERCISE_DIR))

from schema import WaterRetentionBasinDocument

print(f"Exercise dir: {EXERCISE_DIR}")
print("Imports OK")

c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\.venv\Lib\site-packages\dllmforge\IE_agent_config.py:36: UserWarning: Field name "schema" in "IEAgentConfig" shadows an attribute in parent "BaseModel"
  class IEAgentConfig(BaseModel):


Exercise dir: c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction
Imports OK


In [2]:
load_dotenv()
if not os.getenv("AZURE_OPENAI_ENDPOINT") and os.getenv("AZURE_OPENAI_API_BASE"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = os.environ["AZURE_OPENAI_API_BASE"]

llm_api = LangchainAPI(model_provider="azure-openai", temperature=0.1)
print("Azure OpenAI ready (", os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"), ")")

Azure OpenAI ready ( gpt-5.2 )


## Step 2: Define extraction schema

The second step is to know what information you want to extract, and define a pydantic schema. The schema is the contract: if a field is missing, the model will not return it. Let's print out the schema to inspect what information we are going to extract.

In [4]:
print("\nWaterRetentionBasin fields:")
from schema import WaterRetentionBasin

for name, info in WaterRetentionBasin.model_fields.items():
    print(f"  - {name}: {info.annotation}")


WaterRetentionBasin fields:
  - name: typing.Optional[str]
  - alternative_names: typing.Optional[typing.List[str]]
  - country: typing.Optional[str]
  - location: typing.Optional[str]
  - river_name: typing.Optional[str]
  - river_basin: typing.Optional[str]
  - catchment: typing.Optional[str]
  - flood_water_storage_volume_m3: typing.Optional[float]
  - storage_volume_total_m3: typing.Optional[float]
  - basin_control_type: typing.Optional[str]
  - dam_height_m: typing.Optional[float]
  - sill_level_m: typing.Optional[float]
  - operating_rules: typing.Optional[str]
  - source_page: typing.Optional[int]
  - source_quote: typing.Optional[str]


## Step 3: Docling preprocess (sections, tables, pages)

`DoclingDocumentProcessor` converts the PDF to tagged markdown: `[PAGE N]`, `[SECTION: …]`, `[TABLE: …]`, `[FIGURE: …]`. Tables stay as markdown grids (not PyMuPDF cell soup). `[PAGE N]` is the PDF sheet index, not the journal page number. `source_page` is filled later from `source_quote`, not by the LLM.

Note: Docling does not only parse PDF text. It runs neural layout models so it can tell paragraphs from tables, figures, and reading order. Those model weights are not in the pip/uv package. On the first `process_document()` call, Docling fetches them from Hugging Face into a local cache. After that, later runs reuse the cache. No Hugging Face download unless the cache is cleared or the model version changes.

In [5]:
pdf_path = EXERCISE_DIR / "data" / "pdfs" / "nhess-15-365-2015.pdf"
output_dir = EXERCISE_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)

doc_processor = DoclingDocumentProcessor(
    input_dir=str(EXERCISE_DIR / "data" / "pdfs"),
    file_pattern="*.pdf",
    output_dir=str(output_dir),
)
processed = doc_processor.process_document(pdf_path)
print(f"File: {pdf_path.name}")
print(
    f"Pages: {processed.metadata.get('page_count')}, "
    f"characters: {processed.metadata.get('text_length')}, "
    f"tables: {processed.metadata.get('num_tables')}"
)
print("--- first 800 characters ---")
print(processed.content[:800])
table_at = processed.content.find("[TABLE:")
if table_at >= 0:
    print("--- Docling table preview ---")
    print(processed.content[table_at:table_at + 600])

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 9476d0d70a6055dbf76f160ddb363a6a
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotron-ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.models.stages.ocr.auto_ocr_model:Auto OCR model selected rapidocr with onnxruntime.
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered layout engines: ['layout_obj

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

INFO:docling.models.inference_engines.object_detection.transformers_engine:Transformers engine ready (device=cpu, dtype=torch.float32)
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2', 'granite_vision_table']
INFO:docling.models.utils.hf_model_download:Fetching model docling-project/docling-models (revision: v2.3.0)...
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
INFO:docling.models.utils.hf_model_download:Model docling-project/docling-models already cached at C:\Users\deng_jg\.cache\huggingface\hub\models--docling-project--docling-models\snapshots\fc0f2d45e2218ea24bce5045f58a389aed16dc23
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document nhess-15-365-2015.pdf
INFO:docling.document_converter:Finis

File: nhess-15-365-2015.pdf
Pages: 15, characters: 63347, tables: 9
--- first 800 characters ---
[PAGE 1]

[SECTION: Assessing the operation rules of a reservoir system based on a detailed modelling chain]

M. Bruwier, S. Erpicum, M. Pirotton, P. Archambeau, and B. J. Dewals

University of Liege (ULG), Hydraulics in Environmental and Civil Engineering (HECE), Liege, Belgium

Correspondence to: M. Bruwier (mbruwier@ulg.ac.be)

Received: 3 July 2014 - Published in Nat. Hazards Earth Syst. Sci. Discuss.: 9 September 2014

Revised: 4 February 2015 - Accepted: 6 February 2015 - Published: 2 March 2015

Abstract. According to available climate change scenarios for Belgium, drier summers and wetter winters are expected. In this study, we focus on two multi-purpose reservoirs located in the Vesdre catchment, which is part of the Meuse basin. The current operation rules of the reservoirs are f
--- Docling table preview ---
[TABLE: Table 1. Summary of the reservoir characteristics.]
Table 2. Sce

## Step 4: Build the extractor

**1) Define the system prompt** for the extraction task. Prefer markdown **table** values over rounded prose; copy one table row or one sentence as `source_quote`; do not set `source_page`.

**2) Reuse the Docling processor** from Step 3 (`doc_processor`).

**3) Create `DoclingInfoExtractor`.** It combines the schema, the LLM, the Docling processor, and the system prompt. After the LLM returns, DLLMForge locates `source_quote` in the element index and fills `source_page`.

In [6]:
system_prompt = """Extract every water retention basin, polder, or multi-purpose reservoir
mentioned in the document into the provided schema.

Rules:
- One list entry per distinct site. Do not merge two dams or basins (e.g. keep Eupen and La Gileppe separate).
- Only extract values that are explicitly stated. Use null if a field is missing or uncertain.
- Convert all volumes to cubic metres (m³). 1 hm³ = 1e6 m³, 1 Mm³ = 1e6 m³.
- river_name is the local river; river_basin is the main basin (e.g. Meuse).
- Prefer markdown table values over rounded prose when they conflict.
- Expand table scientific notation to a number (25 × 10^6 m³ → 25000000).
- Copy source_quote from one table row or one sentence (not several blocks concatenated). Do not paraphrase.
- Do not set source_page; it is filled after extraction from the quote.

Return JSON only, matching the schema keys.
"""

extractor = DoclingInfoExtractor(
    output_schema=WaterRetentionBasinDocument,
    llm_api=llm_api,
    system_prompt=system_prompt,
    chunk_size=80000,
    chunk_overlap=10000,
    doc_processor=doc_processor,
)
print("Extractor ready")

Extractor ready


## Step 5: Extract from one PDF

This cell calls Azure GPT and may take a minute.

In [7]:
results = extractor.process_document(processed)
output_path = output_dir / f"{pdf_path.stem}_extracted.json"
extractor.save_results(results, output_path)

print(f"Saved {len(results)} result object(s) to {output_path}")
print(json.dumps(results, indent=2, ensure_ascii=False, default=str))

INFO:httpx:HTTP Request: POST https://openai-dllmforge.openai.azure.com/openai/deployments/gpt-5.2/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Results saved to c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction\output\nhess-15-365-2015_extracted.json
Saved 1 result object(s) to c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\01_information_extraction\output\nhess-15-365-2015_extracted.json
[
  {
    "source_title": "Assessing the operation rules of a reservoir system based on a detailed modelling chain",
    "basins": [
      {
        "name": "Eupen dam",
        "alternative_names": [
          "Eupen reservoir"
        ],
        "country": "Belgium",
        "location": "3km upstream of the town Eupen",
        "river_name": "Vesdre and Getzbach",
        "river_basin": "Meuse",
        "catchment": "Vesdre catchment",
        "flood_water_storage_volume_m3": null,
        "storage_volume_total_m3": 25000000.0,
        "basin_control_type": "cont

## Step 6: Inspect metadata vs evidence, then check 'groundtruth'values

- `metadata` comes from the Docling processor (file name, page count, table/figure counts). The locator index is used internally and is not written to the JSON.
- `source_quote` comes from the LLM. `source_page` / `quote_verified` are filled by DLLMForge from that quote. Open the PDF sheet and confirm the quote is there.

Groundtruth values below are **approximate** (Bruwier et al., 2015). Volumes may be wrong if the model forgot to convert hm³ / Mm³ to m³.

In [8]:
basins = []
for result in results:
    print("Processor metadata:", result.get("metadata"))
    basins.extend(result.get("basins") or [])

print(f"\nExtracted {len(basins)} basin(s)")
for basin in basins:
    print("-" * 60)
    print(f"name:         {basin.get('name')}")
    print(f"country:      {basin.get('country')}")
    print(f"river_name:   {basin.get('river_name')}")
    print(f"river_basin:  {basin.get('river_basin')}")
    print(f"control:      {basin.get('basin_control_type')}")
    print(f"total m3:     {basin.get('storage_volume_total_m3')}")
    print(f"flood m3:     {basin.get('flood_water_storage_volume_m3')}")
    print(f"source_page:  {basin.get('source_page')}")
    print(f"quote_ok:     {basin.get('quote_verified')}")
    print(f"source_quote: {basin.get('source_quote')}")

GOLD_VALUES = [
    {"name": "Eupen dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
    {"name": "La Gileppe dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
]
print("\nGold-value check:")
print(f"{'expected':<22} {'extracted name':<28} {'country':<12} {'river':<12} {'type':<12}")
for gold in GOLD_VALUES:
    needle = gold["name"].split()[0].lower()
    match = next((b for b in basins if needle in (b.get("name") or "").lower()), {})
    print(
        f"{gold['name']:<22} "
        f"{str(match.get('name') or 'NOT FOUND'):<28} "
        f"{str(match.get('country') or ''):<12} "
        f"{str(match.get('river_name') or ''):<12} "
        f"{str(match.get('basin_control_type') or ''):<12}"
    )

Processor metadata: {'source_file': 'c:\\Users\\deng_jg\\work\\16centralized_agents\\hydrology_user_day_training\\LLMs-Agentic-Workflows-in-Hydrology-Training\\exercises\\01_information_extraction\\data\\pdfs\\nhess-15-365-2015.pdf', 'file_name': 'nhess-15-365-2015.pdf', 'file_type': '.pdf', 'text_length': 63347, 'page_count': 15, 'num_pages': 15, 'num_tables': 9, 'num_figures': 11, 'processing_method': 'docling'}

Extracted 2 basin(s)
------------------------------------------------------------
name:         Eupen dam
country:      Belgium
river_name:   Vesdre and Getzbach
river_basin:  Meuse
control:      controlled
total m3:     25000000.0
flood m3:     None
source_page:  4
quote_ok:     True
source_quote: | Capacity                                    | 25 × 10^6 hm 3      | 26.4 × 10^6 hm 3 |
------------------------------------------------------------
name:         La Gileppe dam
country:      Belgium
river_name:   La Gileppe
river_basin:  Meuse
control:      controlled
total m3: 